In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import os
import numpy as np
import json
from pathlib import Path
from sentence_transformers import SentenceTransformer
from datetime import datetime, timezone

In [2]:
USER_AGENT = os.getenv(
    "USER_AGENT",
    "TourGuideAI/1.0 (Learning project), Mozilla/5.0 (Windows NT 10.0; Win64; x64), Chrome/91.0.4472.124 Safari/537.36")

In [3]:
BASE = "https://www.spreewald-info.de"
headers = {"User-Agent": USER_AGENT}

# Links sammeln: Es gibt mehrere Seiten mit Bootsverleih, die alle unter "/paddeln/bootsverleih/" liegen.
url = BASE + "/paddeln/bootsverleih/"
resp = requests.get(url, headers=headers, timeout=20)
resp.raise_for_status()
html = resp.text
soup = BeautifulSoup(html, "html.parser")

links = []

for a in soup.select("a"): #alle Links auf der Seite durchgehen
    href = a.get("href")
    if href.startswith("/paddeln/bootsverleih/") and href != "/paddeln/bootsverleih/":
        links.append(urljoin(BASE, href))
    # if href and "/paddeln/bootsverleih/" in href and href.count("/") >= 3:
    #     links.append(urljoin(BASE, href))

links = list(set(links)) #Duplikate entfernen

for i, link in enumerate(links, start=1):
     print(f"{i}. {link}")


1. https://www.spreewald-info.de/paddeln/bootsverleih/bootsverleih-hafen-zur-alten-aalreuse
2. https://www.spreewald-info.de/paddeln/bootsverleih/bootshaus-conrad
3. https://www.spreewald-info.de/paddeln/bootsverleih/bootsverleih-keutel
4. https://www.spreewald-info.de/paddeln/bootsverleih/bootshaus-leineweber
5. https://www.spreewald-info.de/paddeln/bootsverleih/bootsverleih-richter
6. https://www.spreewald-info.de/paddeln/bootsverleih/kleiner-hafen
7. https://www.spreewald-info.de/paddeln/bootsverleih/stand-up-paddling-spreewald
8. https://www.spreewald-info.de/paddeln/bootsverleih/bootsvermietung-henschelchen-schlepzig
9. https://www.spreewald-info.de/paddeln/bootsverleih/spreehafen-burg
10. https://www.spreewald-info.de/paddeln/bootsverleih/spreewald-kanus
11. https://www.spreewald-info.de/paddeln/bootsverleih/bootsverleih-jentsch
12. https://www.spreewald-info.de/paddeln/bootsverleih/bootsverleih-gromsch
13. https://www.spreewald-info.de/paddeln/bootsverleih/bootsverleih-gasthof-z

In [4]:
# Daten extrahieren
def parse_prices(url):

    resp = requests.get(url, headers=headers, timeout=20)
    resp.raise_for_status()
    html = resp.text
    soup = BeautifulSoup(html, "html.parser")

    title_tag = soup.find("h1")
    title = title_tag.get_text(strip=True) if title_tag else "Unknown"

    prices = []

    for li in soup.select(".anbieterDetailsInfobox li"):
        text = " ".join(li.stripped_strings)
        if "Euro" in text or "€" in text:
            prices.append(text)

    return {
        "anbieter": title,
        "url": url,
        "preise": prices
    }


def fetch_prices_live(source_url: str) -> dict:
    # Holt aktuelle Preise live von der Quellseite
    return parse_prices(source_url)


In [5]:
# Alle Anbieter und Preise sammeln. parce_prices() ist widerverwendbar
docs = []

for link in links:
    try:
        data = parse_prices(link)
        docs.append(data)

        print("\nAnbieter:", data["anbieter"])
        print("URL:", data["url"])

        for p in data["preise"]:
            print("  -", p)

    except Exception as e:
        print("Fehler bei:", link)
        print(e)


print("\nGesamt Anbieter gescraped:", len(docs))


Anbieter: Bootsverleih am Hafen zur alten Aalreuse
URL: https://www.spreewald-info.de/paddeln/bootsverleih/bootsverleih-hafen-zur-alten-aalreuse

Anbieter: Bootshaus Conrad - Bootsverleih in Burg
URL: https://www.spreewald-info.de/paddeln/bootsverleih/bootshaus-conrad
  - 1-Sitzer Paddelboot / Kajak 3 Stunden 20,00 euro | Tagespreis 25,00 Euro
  - 2-Sitzer Paddelboot / Kajak 3 Stunden 30,00 Euro | Tagespreis 35,00 Euro
  - 3-Sitzer Paddelboot / Kajak 3 Stunden 35,00 Euro | Tagespreis 40,00 Euro
  - 3-Sitzer Kanu / Canadier 2 Stunden 35,00 Euro | Tagespreis 40,00 Euro
  - 4-Sitzer Kanu / Canadier 3 Stunden 40,00 Euro | Tagespreis 45,00 Euro

Anbieter: Bootshaus Keutel
URL: https://www.spreewald-info.de/paddeln/bootsverleih/bootsverleih-keutel

Anbieter: Bootsverleih am Bootshaus Leineweber
URL: https://www.spreewald-info.de/paddeln/bootsverleih/bootshaus-leineweber
  - 1er Kajak Preis 2 Stunden: 20,00 Euro Tagespreis 28,00 Euro
  - 2er Kajak Preis 2 Stunden: 26,00 Euro Tagespreis 37,00

In [6]:
#JSON für RAG vorbereiten. Die Preise können sich ändern, deswegen werden nicht direckt in den Text stehen

output_path = Path("providers.jsonl")
retrieved_at = datetime.now(timezone.utc).isoformat()

with output_path.open("w", encoding="utf-8") as f:
    for item in docs:
       #text feld für Embeddings vorbereiten
        text_parts = []
        anbieter = item.get("anbieter", "")
        preise = item.get("preise", "")
        source_url = item.get("url", "")
        text = (
            f"Anbieter: {anbieter}\n"
            f"Leistung: Bootsverleih\n"
            f"Hinweis: Aktuelle Preise bitte über die Quelle abrufen."
        ).strip()

        doc_entry = {
            "id": source_url,
            "text": text,
            "metadata": {
                "source_url": source_url,
                "anbieter": anbieter,
                "license": "unknown",
                "retrieved_at": retrieved_at,
            }
        }

        f.write(json.dumps(doc_entry, ensure_ascii=False) + "\n")


In [7]:
#test
if docs:
    sample = docs[0]
    live = fetch_prices_live(sample["url"])
    print("\n Anbieter:", live.get("anbieter"))
    print(" Preise :")
    for p in (live.get("preise") or []):
        print(" -", p)



 Anbieter: Bootsverleih am Hafen zur alten Aalreuse
 Preise :


In [8]:
# embedding
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

data = []

with open("providers.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        data.append(item)
print(f"{len(data)} Dokumente geladen.")

17 Dokumente geladen.


In [10]:
texts = [item['text'] for item in data]

embeddings_preise = model.encode(texts, show_progress_bar=True)
embeddings = np.array(embeddings_preise).astype('float32')
print(f"Embeddings für {len(embeddings)} Dokumente erstellt.")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings für 17 Dokumente erstellt.
